In [ ]:
# data-agent notebook bootstrap
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

try:
    from IPython.display import display
except Exception:
    def display(value):
        print(value)

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
try:
    from IPython.display import display
except ImportError:
    def display(x): print(x)

%matplotlib inline

In [ ]:
# Load the unified dataset
data_path = Path("unified_dataset.jsonl")
df = pd.read_json(data_path, lines=True)

print(f"Dataset shape: {df.shape}")
print(f"\nColumn names: {list(df.columns)}")
print("\n--- First 5 rows ---")
display(df.head())

In [ ]:
# Schema and data types
print("--- Data Types ---")
print(df.dtypes)

print("\n--- Basic Info ---")
df.info()

In [ ]:
# Missingness analysis
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
print("--- Missing Values ---")
display(missing_df[missing_df["missing_count"] > 0])

# Visualize missingness
fig, ax = plt.subplots(figsize=(8, 4))
missing_df["missing_pct"].plot(kind="bar", ax=ax, color="coral")
ax.set_title("Missing Value Percentage by Column")
ax.set_ylabel("Missing %")
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Label distribution
if "label" in df.columns and df["label"].notna().sum() > 0:
    label_counts = df["label"].value_counts()
    print(f"--- Label Distribution ---")
    print(f"Unique labels: {df['label'].nunique()}")
    print(f"Non-null labels: {df['label'].notna().sum()}")
    
    fig, ax = plt.subplots(figsize=(10, 5))
    label_counts.head(20).plot(kind="bar", ax=ax, color="steelblue")
    ax.set_title("Label Distribution (Top 20)")
    ax.set_ylabel("Count")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("No label column or all labels are null.")

In [ ]:
# Text length distribution
if "text" in df.columns and df["text"].notna().sum() > 0:
    df["text_length"] = df["text"].str.split().str.len()
    
    print("--- Text Length Statistics (words) ---")
    print(df["text_length"].describe())
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Histogram
    axes[0].hist(df["text_length"], bins=30, color="teal", edgecolor="black")
    axes[0].set_title("Text Length Distribution")
    axes[0].set_xlabel("Word Count")
    axes[0].set_ylabel("Frequency")
    
    # Boxplot
    axes[1].boxplot(df["text_length"], vert=True)
    axes[1].set_title("Text Length Boxplot")
    axes[1].set_ylabel("Word Count")
    
    plt.tight_layout()
    plt.show()
else:
    print("No text column or all texts are null.")

In [ ]:
# Source distribution
if "source" in df.columns:
    source_counts = df["source"].value_counts()
    print("--- Source Distribution ---")
    print(source_counts)
    
    fig, ax = plt.subplots(figsize=(8, 4))
    source_counts.plot(kind="bar", ax=ax, color="darkgreen")
    ax.set_title("Data Source Distribution")
    ax.set_ylabel("Count")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("No source column found.")

In [ ]:
# Image column analysis
if "image" in df.columns:
    image_count = df["image"].notna().sum()
    print(f"--- Image Column ---")
    print(f"Non-null image entries: {image_count}")
    if image_count > 0:
        print("\nSample image values:")
        display(df[df["image"].notna()]["image"].head())
else:
    print("No image column found.")

In [ ]:
# Metadata analysis
if "metadata" in df.columns and df["metadata"].notna().sum() > 0:
    # Extract metadata keys
    metadata_keys = df["metadata"].dropna().apply(lambda x: list(x.keys()) if isinstance(x, dict) else []).explode().unique()
    print("--- Metadata Keys ---")
    print(f"Keys found: {list(metadata_keys)}")
    
    # Check for cleaning_status if present
    if "cleaning_status" in metadata_keys:
        cleaning_counts = df["metadata"].dropna().apply(lambda x: x.get("cleaning_status") if isinstance(x, dict) else None).value_counts()
        print("\n--- Cleaning Status ---")
        print(cleaning_counts)
else:
    print("No metadata column or all metadata is null.")

In [ ]:
# Summary statistics
print("=" * 50)
print("EDA SUMMARY")
print("=" * 50)
print(f"Total rows: {len(df)}")
print(f"Total columns: {len(df.columns)}")
print(f"Text column: {'Present' if 'text' in df.columns else 'Missing'}")
print(f"Label column: {'Present' if 'label' in df.columns else 'Missing'} ({df['label'].notna().sum()} non-null)")
print(f"Source column: {'Present' if 'source' in df.columns else 'Missing'}")
print(f"Image column: {'Present' if 'image' in df.columns else 'Missing'} ({df['image'].notna().sum()} non-null)")
print(f"Audio column: {'Present' if 'audio' in df.columns else 'Missing'} ({df['audio'].notna().sum()} non-null)")
print("=" * 50)

## Data Quality Review

### Analyzer View

- Task interpretation: unknown
- Primary modality: text
- Target semantics: unknown
- Relevant checks: text_non_empty - all 100 rows have valid text (min length 84), label_format_validity - labels follow '#### <number>' format with reasoning chain, source_distribution - madrylab/gsm8k-platinum (50), all-russian (40), project-euler (10), problem_diversity - different math problem types across sources
- Lower-value checks: class_balance plots - not applicable for numeric answer targets with high cardinality, numeric outlier detection - labels are solution text, not numeric features, imputation for missing labels - labels are structured solution chains, not simple numeric values to impute
- Priority actions: drop_irrelevant_columns, preserve_unlabeled_rows_for_evaluation, validate_label_format_consistency

### Strategy Justification

All 100 rows contain valid math problems. Dropped audio/image columns (100%/90% null). No duplicates found. Missing labels (50 rows) are intentional dataset characteristics from different sources, not quality issues - preserved for evaluation/future labeling. Median imputation and IQR outlier clipping are inappropriate for text-based math reasoning dataset where labels are solution chains.

- Missing values: `not_applicable`
- Duplicates: `drop`
- Outliers: `not_applicable`

### Findings

- Missing values before cleaning: 240
- Duplicate rows before cleaning: 0
- Numeric outliers before cleaning: 0
- Imbalance column: `label`
- Majority class share after cleaning: not applicable

### Before / After

- Missing values: 0 -> 0
- Duplicates: 0 -> 0
- Outliers: 0 -> 0


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

raw_df = pd.read_json('data/collection/unified_dataset.jsonl', lines=True)
clean_df = pd.read_json('data/quality/cleaned_dataset.jsonl', lines=True)
primary_modality = 'text'

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
missing_counts = raw_df.isna().sum().sort_values(ascending=False)
missing_counts = missing_counts[missing_counts > 0]
if not missing_counts.empty:
    sns.barplot(x=missing_counts.values, y=missing_counts.index, ax=axes[0, 0], color='#d97706')
    axes[0, 0].set_title('Missing values by column')
else:
    axes[0, 0].text(0.5, 0.5, 'No missing values', ha='center', va='center')
    axes[0, 0].set_axis_off()

if {'source', 'label'}.issubset(raw_df.columns):
    coverage = raw_df.assign(label_present=raw_df['label'].notna()).groupby('source', dropna=False)['label_present'].mean().sort_values(ascending=False).head(10)
    if not coverage.empty:
        sns.barplot(x=coverage.values, y=coverage.index.astype(str), ax=axes[0, 1], color='#2563eb')
        axes[0, 1].set_title('Label coverage by source')
        axes[0, 1].set_xlim(0, 1)
    else:
        axes[0, 1].text(0.5, 0.5, 'No source coverage data', ha='center', va='center')
        axes[0, 1].set_axis_off()
elif 'source' in raw_df.columns:
    source_counts = raw_df['source'].astype(str).value_counts().head(10)
    sns.barplot(x=source_counts.values, y=source_counts.index, ax=axes[0, 1], color='#2563eb')
    axes[0, 1].set_title('Top sources')
else:
    axes[0, 1].text(0.5, 0.5, 'No source column', ha='center', va='center')
    axes[0, 1].set_axis_off()

numeric_columns = raw_df.select_dtypes(include=['number']).columns.tolist()
if numeric_columns and not (len(numeric_columns) == 1 and 'label' in numeric_columns and primary_modality == 'text'):
    sns.boxplot(data=raw_df[numeric_columns], orient='h', ax=axes[1, 0], color='#f59e0b')
    axes[1, 0].set_title('Raw numeric distributions')
    sns.boxplot(data=clean_df[numeric_columns], orient='h', ax=axes[1, 1], color='#10b981')
    axes[1, 1].set_title('Cleaned numeric distributions')
else:
    if 'text' in raw_df.columns:
        raw_lengths = raw_df['text'].fillna('').astype(str).str.split().str.len()
        clean_lengths = clean_df['text'].fillna('').astype(str).str.split().str.len()
        sns.histplot(raw_lengths, bins=30, ax=axes[1, 0], color='#f59e0b')
        axes[1, 0].set_title('Raw text length distribution')
        sns.histplot(clean_lengths, bins=30, ax=axes[1, 1], color='#10b981')
        axes[1, 1].set_title('Cleaned text length distribution')
    else:
        axes[1, 0].text(0.5, 0.5, 'No numeric columns', ha='center', va='center')
        axes[1, 0].set_axis_off()
        axes[1, 1].text(0.5, 0.5, 'No numeric columns', ha='center', va='center')
        axes[1, 1].set_axis_off()

plt.tight_layout()
plt.show()
